In [ ]:
import subprocess, sys, importlib.metadata
try:
    v = importlib.metadata.version('timm')
except Exception:
    v = None
if v != '1.0.22':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'timm==1.0.22'])


In [ ]:
from pathlib import Path
import sys
Path('raptor_v5_radimagenet.py').write_text('"""Raptor v5 with residual RadImageNet fusion inside CoAtNet.\n\nInput is an already preprocessed [B, K, 3, 384, 384] triplet bag. Both\nencoders receive exactly the same tensor; preprocessing belongs to the loader.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nfrom torch import Tensor, nn\nfrom torch.utils.checkpoint import checkpoint\n\n\nCOATNET_ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"\nLABELS = (\n    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",\n    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s",\n    "Contusion", "Fracture",\n)\n\n\n@dataclass(frozen=True)\nclass FusionConfig:\n    stage2_hidden: int = 1024\n    stage4_hidden: int = 2048\n    mlp_dropout: float = 0.0\n    head_dropout: float = 0.2\n    gradient_checkpointing: bool = False\n\n\nclass ConcatenationMLP(nn.Module):\n    """Three affine layers with GELU, applied to each spatial position.\n\n    Concatenation precedes every learned operation. Only the output projection\n    is zero-initialized, so the residual is exactly zero without a dead MLP.\n    """\n\n    def __init__(self, first_channels: int, second_channels: int,\n                 out_channels: int, hidden: int, dropout: float = 0.0):\n        super().__init__()\n        if min(first_channels, second_channels, out_channels, hidden) <= 0:\n            raise ValueError("MLP channel dimensions must be positive")\n        self.first_channels = first_channels\n        self.second_channels = second_channels\n        self.mlp = nn.Sequential(\n            nn.LayerNorm(first_channels + second_channels),\n            nn.Linear(first_channels + second_channels, hidden),\n            nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, hidden),\n            nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, out_channels),\n        )\n        nn.init.zeros_(self.mlp[-1].weight)\n        nn.init.zeros_(self.mlp[-1].bias)\n\n    def forward(self, first: Tensor, second: Tensor) -> Tensor:\n        if (first.ndim != 4 or second.ndim != 4\n                or first.shape[0] != second.shape[0]\n                or first.shape[2:] != second.shape[2:]\n                or first.shape[1] != self.first_channels\n                or second.shape[1] != self.second_channels):\n            raise ValueError(f"Incompatible fusion features: {first.shape}, {second.shape}")\n        joined = torch.cat((first, second), dim=1).permute(0, 2, 3, 1)\n        return self.mlp(joined).permute(0, 3, 1, 2).contiguous()\n\n\nclass RadImageNetEncoder(nn.Module):\n    """The same torchvision ResNet-50 trunk used by the baseline Rad encoder.\n\n    Keep `backbone.0` ... `backbone.7` names for strict ResNet50.pt loading.\n    conv3_x is backbone[5] (torchvision layer2); conv5_x is backbone[7]\n    (layer4). No ImageNet weights are substituted for RadImageNet weights.\n    """\n\n    def __init__(self):\n        super().__init__()\n        from torchvision.models import resnet50\n\n        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])\n\n    def forward(self, image: Tensor) -> tuple[Tensor, Tensor]:\n        x = image\n        conv3 = None\n        for index, layer in enumerate(self.backbone):\n            x = layer(x)\n            if index == 5:\n                conv3 = x\n        return conv3, x\n\n\nclass FusedCoAtNet(nn.Module):\n    """Stage2 Z += F1(X,Z), Stage4 V += F2(V,Y), then normal CoAtNet pooling."""\n\n    def __init__(self, coatnet: nn.Module, radimagenet: nn.Module,\n                 config: FusionConfig = FusionConfig()):\n        super().__init__()\n        self.coatnet = coatnet\n        self.radimagenet = radimagenet\n        self.num_features = coatnet.num_features\n        channels = [entry["num_chs"] for entry in coatnet.feature_info]\n        if len(coatnet.stages) != 4 or channels[-4:] != [128, 256, 512, 1024]:\n            raise ValueError("Expected the four-stage Raptor v5 CoAtNet RMLP-2")\n        self.f1 = ConcatenationMLP(512, 256, 256, config.stage2_hidden, config.mlp_dropout)\n        self.f2 = ConcatenationMLP(1024, 2048, 1024, config.stage4_hidden, config.mlp_dropout)\n        self.gradient_checkpointing = config.gradient_checkpointing\n        self.checkpoint_parts: frozenset[str] = frozenset()\n        self.requires_grad_(True)\n        self.coatnet.set_grad_checkpointing(config.gradient_checkpointing)\n\n    CHECKPOINT_PARTS = ("rad", "s1", "s2", "s3", "s4", "fusion")\n\n    def set_checkpoint_parts(self, parts) -> None:\n        """Runtime-only activation recomputation for selected whole sections.\n\n        Parts: rad (ResNet-50), s1 (stem+S1), s2, s3, s4 (CoAtNet stages) and\n        fusion (both MLPs). Not stored in checkpoints; numerics are unchanged.\n        """\n        parts = frozenset(parts)\n        unknown = parts - set(self.CHECKPOINT_PARTS)\n        if unknown:\n            raise ValueError(f"Unknown checkpoint parts: {sorted(unknown)}")\n        self.checkpoint_parts = parts\n\n    def _run(self, part: str, fn, *args):\n        if part in self.checkpoint_parts and self.training and torch.is_grad_enabled():\n            return checkpoint(fn, *args, use_reentrant=False)\n        return fn(*args)\n\n    def _stem_s1(self, image: Tensor) -> Tensor:\n        return self.coatnet.stages[0](self.coatnet.stem(image))\n\n    def _fuse(self, mlp: nn.Module, first: Tensor, second: Tensor) -> Tensor:\n        if ((self.gradient_checkpointing or "fusion" in self.checkpoint_parts)\n                and self.training and torch.is_grad_enabled()):\n            return checkpoint(mlp, first, second, use_reentrant=False)\n        return mlp(first, second)\n\n    def forward_features(self, image: Tensor) -> Tensor:\n        if image.ndim != 4 or tuple(image.shape[1:]) != (3, 384, 384):\n            raise ValueError("Both backbones require [N, 3, 384, 384] preprocessed triplets")\n        # Feed the identical image object to both trunks; no second normalization.\n        x, y = self._run("rad", self.radimagenet, image)\n        z = self._run("s1", self._stem_s1, image)\n        z = self._run("s2", self.coatnet.stages[1], z)  # S2 convolution, [N,256,48,48]\n        z = z + self._fuse(self.f1, x, z)  # F1 input order is (X,Z)\n        v = self._run("s3", self.coatnet.stages[2], z)  # S3 receives the UPDATED Z\n        v = self._run("s4", self.coatnet.stages[3], v)  # S4 transformer, [N,1024,12,12]\n        v = v + self._fuse(self.f2, v, y)  # F2 input order is (V,Y)\n        return self.coatnet.norm(v)\n\n    def forward(self, image: Tensor) -> Tensor:\n        return self.coatnet.forward_head(self.forward_features(image))\n\n\nclass RaptorClassifier(nn.Module):\n    """Original Raptor attention-MIL head, retaining checkpoint parameter names."""\n\n    def __init__(self, backbone: nn.Module, F_dim: int, n: int = 12, drop: float = 0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x: Tensor) -> Tensor:\n        if x.ndim != 5 or x.shape[0] == 0 or x.shape[1] == 0:\n            raise ValueError("Raptor expects a nonempty [B,K,3,384,384] triplet bag")\n        b, k = x.shape[:2]\n        return self.backbone(x.flatten(0, 1)).reshape(b, k, -1)\n\n    def head(self, feats: Tensor) -> Tensor:\n        h = self.norm(feats)\n        a = torch.softmax(self.att(h), dim=1)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        return (pooled * self.clsW).sum(-1) + self.clsb\n\n    def forward(self, x: Tensor) -> Tensor:\n        return self.head(self.encode(x))\n\n\ndef _checkpoint(source: str | Path | Mapping[str, Any]) -> Mapping[str, Any]:\n    return source if isinstance(source, Mapping) else torch.load(\n        source, map_location="cpu", weights_only=True,\n    )\n\n\ndef build_raptor_v5_radimagenet(\n    raptor_checkpoint: str | Path | Mapping[str, Any],\n    radimagenet_checkpoint: str | Path | Mapping[str, Any],\n    config: FusionConfig = FusionConfig(),\n) -> RaptorClassifier:\n    """Strict-load both pretrained parents, add zero residuals, unfreeze all.\n\n    Raptor checkpoint: {\'model\': state_dict, \'arch\': ..., \'res\': 384}.\n    Rad checkpoint: the baseline ResNet50.pt state dict, or {\'state_dict\': ...}.\n    Load and resume a *fused* checkpoint using load_fused_checkpoint instead.\n    """\n    import timm\n\n    ck = _checkpoint(raptor_checkpoint)\n    if ck.get("arch", COATNET_ARCH) != COATNET_ARCH or int(ck.get("res", 384)) != 384:\n        raise ValueError("Parent checkpoint must be Raptor v5 CoAtNet RMLP-2 at res=384")\n    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")\n    model = RaptorClassifier(coatnet, F_dim=coatnet.num_features, drop=config.head_dropout)\n    model.load_state_dict(ck["model"], strict=True)\n    rad = RadImageNetEncoder()\n    rad_ck = _checkpoint(radimagenet_checkpoint)\n    rad.load_state_dict(rad_ck.get("state_dict", rad_ck), strict=True)\n    model.backbone = FusedCoAtNet(coatnet, rad, config)\n    model.fusion_config = config\n    model.requires_grad_(True)\n    return model\n\n\ndef optimizer_groups(model: RaptorClassifier, backbone_lr: float = 1e-5,\n                     fusion_lr: float = 1e-4, head_lr: float = 1e-4) -> list[dict[str, Any]]:\n    """All parameters, exactly once, with both backbones unfrozen."""\n    if not isinstance(model.backbone, FusedCoAtNet):\n        raise TypeError("Expected the fused Raptor model")\n    if any(not p.requires_grad for p in model.parameters()):\n        raise ValueError("Both backbones and the fusion/head must remain unfrozen")\n    groups = {"coatnet": [], "radimagenet": [], "fusion": [], "head": []}\n    for name, parameter in model.named_parameters():\n        if name.startswith("backbone.coatnet."):\n            group = "coatnet"\n        elif name.startswith("backbone.radimagenet."):\n            group = "radimagenet"\n        elif name.startswith(("backbone.f1.", "backbone.f2.")):\n            group = "fusion"\n        else:\n            group = "head"\n        groups[group].append(parameter)\n    rates = dict(coatnet=backbone_lr, radimagenet=backbone_lr, fusion=fusion_lr, head=head_lr)\n    return [{"name": name, "params": parameters, "lr": rates[name]}\n            for name, parameters in groups.items()]\n\n\ndef fused_checkpoint(model: RaptorClassifier) -> dict[str, Any]:\n    """Serialize model/config; trainers can add optimizer, scaler and epoch."""\n    return {"format": "raptor_v5_radimagenet_v1", "arch": COATNET_ARCH, "res": 384,\n            "labels": list(LABELS), "fusion_config": asdict(model.fusion_config),\n            "model": model.state_dict()}\n\n\ndef load_fused_checkpoint(source: str | Path | Mapping[str, Any]) -> RaptorClassifier:\n    import timm\n\n    ck = _checkpoint(source)\n    if (ck.get("format") != "raptor_v5_radimagenet_v1" or ck.get("arch") != COATNET_ARCH\n            or ck.get("res") != 384 or ck.get("labels") != list(LABELS)):\n        raise ValueError("Unsupported fused checkpoint architecture/input/label contract")\n    config = FusionConfig(**ck["fusion_config"])\n    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")\n    backbone = FusedCoAtNet(coatnet, RadImageNetEncoder(), config)\n    model = RaptorClassifier(backbone, F_dim=backbone.num_features, drop=config.head_dropout)\n    model.fusion_config = config\n    model.load_state_dict(ck["model"], strict=True)\n    model.requires_grad_(True)\n    return model\n', encoding='utf-8')
sys.path.insert(0, '.')


In [ ]:
exec(compile("\nimport json, time\nimport torch, timm\nfrom pathlib import Path\nfrom raptor_v5_radimagenet import (COATNET_ARCH, FusionConfig, FusedCoAtNet, RadImageNetEncoder, RaptorClassifier)\ntorch.backends.cudnn.benchmark = True\ndev = 'cuda:0'\nprint(torch.__version__, torch.backends.cudnn.version(), torch.cuda.get_device_name(0), flush=True)\ndef build(cl):\n    coat = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, global_pool='avg')\n    bb = FusedCoAtNet(coat, RadImageNetEncoder(), FusionConfig())\n    m = RaptorClassifier(bb, bb.num_features).to(dev).train()\n    if cl: m = m.to(memory_format=torch.channels_last)\n    for mod in m.modules():\n        if isinstance(mod, torch.nn.modules.batchnorm._BatchNorm): mod.eval()\n    return m\ndef run(m, dtype, cl, n=12, train=True, part='full', iters=3):\n    x = torch.randn(n, 3, 384, 384, device=dev)\n    if cl: x = x.contiguous(memory_format=torch.channels_last)\n    def step():\n        with torch.autocast('cuda', dtype=dtype or torch.float16, enabled=dtype is not None):\n            if part == 'coat': out = m.backbone.coatnet(x).float().sum()\n            elif part == 'rad': out = m.backbone.radimagenet(x)[1].float().sum()\n            else: out = m.head(m.backbone(x)[None]).float().sum()\n        if train:\n            out.backward(); m.zero_grad(set_to_none=True)\n    with torch.set_grad_enabled(train):\n        step(); step()\n        torch.cuda.synchronize(); t = time.time()\n        for _ in range(iters): step()\n        torch.cuda.synchronize()\n    return round((time.time()-t)/iters, 3)\nres = []\nfor cl in (False, True):\n    m = build(cl)\n    for c in [dict(dtype=torch.float16), dict(dtype=torch.bfloat16), dict(dtype=None),\n              dict(dtype=torch.float16, part='coat'), dict(dtype=torch.float16, part='rad'),\n              dict(dtype=torch.bfloat16, part='coat'),\n              dict(dtype=torch.float16, train=False), dict(dtype=torch.bfloat16, train=False),\n              dict(dtype=torch.float16, n=4)]:\n        rec = dict(channels_last=cl, **{k: str(v) for k, v in c.items()})\n        try:\n            torch.cuda.reset_peak_memory_stats(0)\n            rec['seconds'] = run(m, cl=cl, **c)\n            rec['peak_gib'] = round(torch.cuda.max_memory_allocated(0)/2**30, 2)\n        except Exception as e:\n            rec['error'] = f'{type(e).__name__}: {str(e)[:160]}'\n            torch.cuda.empty_cache()\n        res.append(rec); print(json.dumps(rec), flush=True)\n    del m; torch.cuda.empty_cache()\nPath('bench.json').write_text(json.dumps(res, indent=1))\n", 'bench', 'exec'))
